# Denial Root-Cause Deep Dive

**Healthcare Revenue Cycle Management (RCM) & Claims Denial Analytics**

This notebook explores synthetic enterprise hospital claims (~12,500 rows) produced by `python run_pipeline.py`.

> **Disclaimer:** Claim-level data is **synthetic**. Patient IDs are hashed. No PHI is present.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..").resolve() if Path("data").exists() is False else Path(".").resolve()
# Prefer repo root whether notebook is opened from notebooks/ or root
for candidate in [Path("."), Path("..")]:
    if (candidate / "data" / "raw_claims.csv").exists() or (candidate / "output" / "claims_denial_kpi_summary.csv").exists():
        ROOT = candidate.resolve()
        break

claims = pd.read_csv(ROOT / "data" / "processed_claims_sanitized.csv", parse_dates=["service_date"])
kpis = pd.read_csv(ROOT / "output" / "claims_denial_kpi_summary.csv")
payer = pd.read_excel(ROOT / "output" / "rcm_executive_metrics.xlsx", sheet_name="Payer_Variance")
pareto = pd.read_excel(ROOT / "output" / "rcm_executive_metrics.xlsx", sheet_name="CARC_Pareto")
recoverable = pd.read_excel(ROOT / "output" / "rcm_executive_metrics.xlsx", sheet_name="Recoverable_by_RootCause")
worklist = pd.read_csv(ROOT / "output" / "appeal_worklist_prioritized.csv")
metrics = pd.read_csv(ROOT / "output" / "appeal_model_metrics.csv")

print(f"Claims: {len(claims):,} | Denied: {(claims.claim_status=='Denied').sum():,}")
kpis

## 1. Executive KPI snapshot

Pipeline-produced metrics (do not hardcode — always read from `output/`).


In [ ]:
kpi_map = dict(zip(kpis["metric"], kpis["value"]))
pd.DataFrame([
    {"KPI": "Gross Denial Rate", "Value": f"{kpi_map['gross_denial_rate_pct']:.2f}%"},
    {"KPI": "Initial Clean Claim Rate", "Value": f"{kpi_map['initial_clean_claim_rate_pct']:.2f}%"},
    {"KPI": "Denied Revenue", "Value": f"${kpi_map['denied_revenue']:,.2f}"},
    {"KPI": "Estimated Recoverable Revenue", "Value": f"${kpi_map['estimated_recoverable_revenue']:,.2f}"},
    {"KPI": "Medicare FFS Observed Denial Rate", "Value": f"{kpi_map['medicare_ffs_observed_denial_rate_pct']:.2f}%"},
    {"KPI": "Appeal Overturn Rate (appealed)", "Value": f"{kpi_map['appeal_overturn_rate_pct']:.2f}%"},
])

## 2. Payer denial variance vs Medicare FFS benchmark (10.8%)


In [ ]:
payer[["payer_name","claims","denied_claims","denial_rate_pct","variance_vs_benchmark_pp","variance_flag"]]

## 3. CARC / RARC root-cause Pareto


In [ ]:
pareto

## 4. Recoverable revenue by root cause


In [ ]:
recoverable.sort_values("estimated_recoverable_revenue", ascending=False)

## 5. Appeal model evaluation & prioritized worklist


In [ ]:
from IPython.display import display
print("Hold-out metrics")
display(metrics)
print("\nTop 15 prioritized appeals")
worklist.head(15)

## 6. Pipeline figures

Figures are written by `src/visualize_rcm.py` to `reports/figures/`.


In [ ]:
from IPython.display import Image, display
fig_dir = ROOT / "reports" / "figures"
for name in [
    "payer_denial_rates_vs_benchmark.png",
    "carc_rarc_root_cause_pareto.png",
    "cumulative_revenue_recovery_curve.png",
]:
    path = fig_dir / name
    print(path.name)
    display(Image(filename=str(path)))